In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# ============================================================
# 00_setup_paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/MLP/Project5"
analysis_dir = os.path.join(PROJECT_ROOT, "analysis_outputs")
os.makedirs(analysis_dir, exist_ok=True)

lazy_exp_dir = os.path.join(
    PROJECT_ROOT,
    "final_outputs",
    "vit_b16_lazystrike_k1_expand_10ep"
)

lazy_eval_dir = os.path.join(
    lazy_exp_dir,
    "eval_jpeg_q30_controlled"
)

orig_path = os.path.join(lazy_eval_dir, "predictions_original.csv")
jpeg_path = os.path.join(lazy_eval_dir, "predictions_jpeg_q30_controlled.csv")

print("analysis_dir:", analysis_dir)
print("lazy_eval_dir:", lazy_eval_dir)
print("orig exists:", os.path.exists(orig_path), orig_path)
print("jpeg exists:", os.path.exists(jpeg_path), jpeg_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
analysis_dir: /content/drive/MyDrive/MLP/Project5/analysis_outputs
lazy_eval_dir: /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_lazystrike_k1_expand_10ep/eval_jpeg_q30_controlled
orig exists: True /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_lazystrike_k1_expand_10ep/eval_jpeg_q30_controlled/predictions_original.csv
jpeg exists: True /content/drive/MyDrive/MLP/Project5/final_outputs/vit_b16_lazystrike_k1_expand_10ep/eval_jpeg_q30_controlled/predictions_jpeg_q30_controlled.csv


In [4]:
# ============================================================
# 01_load_lazystrike_predictions
# ============================================================

orig = pd.read_csv(orig_path)
jpeg = pd.read_csv(jpeg_path)

print("orig shape:", orig.shape)
print("jpeg shape:", jpeg.shape)

print("orig columns:")
print(orig.columns.tolist())

display(orig.head())
display(jpeg.head())

orig shape: (12005, 14)
jpeg shape: (12005, 14)
orig columns:
['image_path', 'filename', 'image_id', 'true_label_idx', 'true_label_name', 'pred_label_idx', 'pred_label_name', 'correct', 'split', 'model_name', 'method_name', 'experiment_name', 'prob_fake', 'prob_real']


,image_path,filename,image_id,true_label_idx,true_label_name,pred_label_idx,pred_label_name,correct,split,model_name,method_name,experiment_name,prob_fake,prob_real
0,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0001.jpg,0001,0,fake,0,fake,True,test_original,vit_b16_lazystrike,LazyStrike-k1,vit_b16_lazystrike_k1_expand_10ep,0.999998,2.348089e-06
1,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0002.jpg,0002,0,fake,0,fake,True,test_original,vit_b16_lazystrike,LazyStrike-k1,vit_b16_lazystrike_k1_expand_10ep,1.000000,4.802962e-08
2,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0003.jpg,0003,0,fake,0,fake,True,test_original,vit_b16_lazystrike,LazyStrike-k1,vit_b16_lazystrike_k1_expand_10ep,0.997353,2.647275e-03
3,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0004.jpg,0004,0,fake,0,fake,True,test_original,vit_b16_lazystrike,LazyStrike-k1,vit_b16_lazystrike_k1_expand_10ep,0.999990,9.930689e-06
4,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0005.jpg,0005,0,fake,0,fake,True,test_original,vit_b16_lazystrike,LazyStrike-k1,vit_b16_lazystrike_k1_expand_10ep,1.000000,1.896144e-07


,image_path,filename,image_id,true_label_idx,true_label_name,pred_label_idx,pred_label_name,correct,split,model_name,method_name,experiment_name,prob_fake,prob_real
0,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0001.jpg,0001,0,fake,0,fake,True,test_jpeg_q30_controlled,vit_b16_lazystrike,LazyStrike-k1,vit_b16_lazystrike_k1_expand_10ep,0.999987,1.271413e-05
1,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0002.jpg,0002,0,fake,0,fake,True,test_jpeg_q30_controlled,vit_b16_lazystrike,LazyStrike-k1,vit_b16_lazystrike_k1_expand_10ep,1.000000,1.135024e-07
2,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0003.jpg,0003,0,fake,0,fake,True,test_jpeg_q30_controlled,vit_b16_lazystrike,LazyStrike-k1,vit_b16_lazystrike_k1_expand_10ep,0.950010,4.999027e-02
3,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0004.jpg,0004,0,fake,0,fake,True,test_jpeg_q30_controlled,vit_b16_lazystrike,LazyStrike-k1,vit_b16_lazystrike_k1_expand_10ep,0.999546,4.535111e-04
4,/content/drive/MyDrive/MLP/Project5/dataset_ex...,0005.jpg,0005,0,fake,0,fake,True,test_jpeg_q30_controlled,vit_b16_lazystrike,LazyStrike-k1,vit_b16_lazystrike_k1_expand_10ep,1.000000,4.715934e-07


In [5]:
# ============================================================
# 02_paired_transition_analysis
# ============================================================

def add_pair_key(df):
    df = df.copy()
    df["pair_key"] = (
        df["true_label_name"].astype(str)
        + "_"
        + df["image_id"].astype(str)
    )
    return df

orig = add_pair_key(orig)
jpeg = add_pair_key(jpeg)

print("Original duplicated pair_key:", orig["pair_key"].duplicated().sum())
print("JPEG duplicated pair_key:", jpeg["pair_key"].duplicated().sum())

orig_small = orig[
    [
        "pair_key",
        "image_path",
        "filename",
        "image_id",
        "true_label_idx",
        "true_label_name",
        "pred_label_idx",
        "pred_label_name",
        "correct",
        "prob_fake",
        "prob_real",
    ]
].copy()

jpeg_small = jpeg[
    [
        "pair_key",
        "image_path",
        "filename",
        "image_id",
        "true_label_idx",
        "true_label_name",
        "pred_label_idx",
        "pred_label_name",
        "correct",
        "prob_fake",
        "prob_real",
    ]
].copy()

paired = orig_small.merge(
    jpeg_small,
    on="pair_key",
    suffixes=("_orig", "_jpeg"),
    how="inner"
)

label_mismatch = (
    paired["true_label_name_orig"] != paired["true_label_name_jpeg"]
).sum()

print("label mismatch:", label_mismatch)
print("paired shape:", paired.shape)

paired["prob_fake_drop"] = paired["prob_fake_orig"] - paired["prob_fake_jpeg"]
paired["prob_fake_change"] = paired["prob_fake_jpeg"] - paired["prob_fake_orig"]
paired["prob_real_change"] = paired["prob_real_jpeg"] - paired["prob_real_orig"]

def classify_transition(row):
    true_label = row["true_label_name_orig"]
    orig_pred = row["pred_label_name_orig"]
    jpeg_pred = row["pred_label_name_jpeg"]

    if true_label == "fake":
        if orig_pred == "fake" and jpeg_pred == "fake":
            return "fake_TP_to_TP"
        elif orig_pred == "fake" and jpeg_pred == "real":
            return "fake_TP_to_FN"
        elif orig_pred == "real" and jpeg_pred == "fake":
            return "fake_FN_to_TP"
        elif orig_pred == "real" and jpeg_pred == "real":
            return "fake_FN_to_FN"

    if true_label == "real":
        if orig_pred == "real" and jpeg_pred == "real":
            return "real_TN_to_TN"
        elif orig_pred == "real" and jpeg_pred == "fake":
            return "real_TN_to_FP"
        elif orig_pred == "fake" and jpeg_pred == "real":
            return "real_FP_to_TN"
        elif orig_pred == "fake" and jpeg_pred == "fake":
            return "real_FP_to_FP"

    return "other"

paired["transition"] = paired.apply(classify_transition, axis=1)

transition_counts = paired["transition"].value_counts().reset_index()
transition_counts.columns = ["transition", "count"]

display(transition_counts)

save_path = os.path.join(
    analysis_dir,
    "vit_b16_lazystrike_k1_paired_predictions_original_vs_jpeg.csv"
)
paired.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved paired predictions:", save_path)

Original duplicated pair_key: 0
JPEG duplicated pair_key: 0
label mismatch: 0
paired shape: (12005, 21)


,transition,count
0,fake_TP_to_TP,5602
1,real_TN_to_TN,5560
2,real_FP_to_FP,323
3,fake_FN_to_FN,250
4,fake_TP_to_FN,136
5,real_FP_to_TN,106
6,real_TN_to_FP,16
7,fake_FN_to_TP,12


Saved paired predictions: /content/drive/MyDrive/MLP/Project5/analysis_outputs/vit_b16_lazystrike_k1_paired_predictions_original_vs_jpeg.csv


In [6]:
# ============================================================
# 03_lazystrike_transition_summary
# ============================================================

fake_subset = paired[paired["true_label_name_orig"] == "fake"]
real_subset = paired[paired["true_label_name_orig"] == "real"]

count_dict = paired["transition"].value_counts().to_dict()

lazy_transition_summary = pd.DataFrame([
    {
        "model_key": "vit_b16_lazystrike_k1",
        "model_name": "ViT-B/16 + LazyStrike-k1",
        "fake_total": len(fake_subset),
        "real_total": len(real_subset),

        "fake_TP_to_TP": count_dict.get("fake_TP_to_TP", 0),
        "fake_TP_to_FN": count_dict.get("fake_TP_to_FN", 0),
        "fake_FN_to_TP": count_dict.get("fake_FN_to_TP", 0),
        "fake_FN_to_FN": count_dict.get("fake_FN_to_FN", 0),

        "real_TN_to_TN": count_dict.get("real_TN_to_TN", 0),
        "real_TN_to_FP": count_dict.get("real_TN_to_FP", 0),
        "real_FP_to_TN": count_dict.get("real_FP_to_TN", 0),
        "real_FP_to_FP": count_dict.get("real_FP_to_FP", 0),

        "fake_TP_to_FN_ratio_percent": count_dict.get("fake_TP_to_FN", 0)
        / len(fake_subset)
        * 100,

        "mean_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].mean(),
        "median_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].median(),
        "std_prob_fake_drop_fake_images": fake_subset["prob_fake_drop"].std(),

        "mean_prob_fake_drop_real_images": real_subset["prob_fake_drop"].mean(),
        "median_prob_fake_drop_real_images": real_subset["prob_fake_drop"].median(),
        "std_prob_fake_drop_real_images": real_subset["prob_fake_drop"].std(),
    }
])

save_path = os.path.join(
    analysis_dir,
    "vit_b16_lazystrike_k1_transition_summary_original_vs_jpeg.csv"
)

lazy_transition_summary.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)
display(lazy_transition_summary)

Saved: /content/drive/MyDrive/MLP/Project5/analysis_outputs/vit_b16_lazystrike_k1_transition_summary_original_vs_jpeg.csv


,model_key,model_name,fake_total,real_total,fake_TP_to_TP,fake_TP_to_FN,fake_FN_to_TP,fake_FN_to_FN,real_TN_to_TN,real_TN_to_FP,real_FP_to_TN,real_FP_to_FP,fake_TP_to_FN_ratio_percent,mean_prob_fake_drop_fake_images,median_prob_fake_drop_fake_images,std_prob_fake_drop_fake_images,mean_prob_fake_drop_real_images,median_prob_fake_drop_real_images,std_prob_fake_drop_real_images
0,vit_b16_lazystrike_k1,ViT-B/16 + LazyStrike-k1,6000,6005,5602,136,12,250,5560,16,106,323,2.266667,0.01998,0.000002,0.099147,0.015186,2.740643e-07,0.085711


In [7]:
# ============================================================
# 04_compare_vit_baseline_vs_lazystrike
# ============================================================

baseline_summary_path = os.path.join(
    analysis_dir,
    "transition_summary_original_vs_jpeg.csv"
)

baseline_transition_summary = pd.read_csv(baseline_summary_path)

vit_baseline = baseline_transition_summary[
    baseline_transition_summary["model_key"] == "vit_b16"
].copy()

compare_df = pd.concat(
    [
        vit_baseline,
        lazy_transition_summary
    ],
    ignore_index=True
)

compare_cols = [
    "model_key",
    "model_name",
    "fake_total",
    "fake_TP_to_FN",
    "fake_FN_to_TP",
    "fake_TP_to_FN_ratio_percent",
    "mean_prob_fake_drop_fake_images",
    "median_prob_fake_drop_fake_images",
]

display(compare_df[compare_cols])

save_path = os.path.join(
    analysis_dir,
    "vit_baseline_vs_lazystrike_paired_comparison.csv"
)

compare_df[compare_cols].to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", save_path)

,model_key,model_name,fake_total,fake_TP_to_FN,fake_FN_to_TP,fake_TP_to_FN_ratio_percent,mean_prob_fake_drop_fake_images,median_prob_fake_drop_fake_images
0,vit_b16,ViT-B/16,6000,102,19,NaN,0.016788,NaN
1,vit_b16_lazystrike_k1,ViT-B/16 + LazyStrike-k1,6000,136,12,2.266667,0.019980,0.000002


Saved: /content/drive/MyDrive/MLP/Project5/analysis_outputs/vit_baseline_vs_lazystrike_paired_comparison.csv


In [8]:
# ============================================================
# Clean comparison table: ViT baseline vs LazyStrike
# ============================================================

clean_compare_df = pd.DataFrame([
    {
        "model_name": "ViT-B/16",
        "fake_total": 6000,
        "fake_TP_to_FN": 102,
        "fake_FN_to_TP": 19,
        "fake_TP_to_FN_ratio_percent": 102 / 6000 * 100,
        "mean_prob_fake_drop_fake_images": 0.016788,
        "interpretation": "Baseline"
    },
    {
        "model_name": "ViT-B/16 + LazyStrike-k1",
        "fake_total": 6000,
        "fake_TP_to_FN": 136,
        "fake_FN_to_TP": 12,
        "fake_TP_to_FN_ratio_percent": 136 / 6000 * 100,
        "mean_prob_fake_drop_fake_images": 0.019980,
        "interpretation": "Higher fake sensitivity, but larger JPEG-induced fake-to-real shift"
    },
])

save_path = os.path.join(
    analysis_dir,
    "vit_baseline_vs_lazystrike_clean_comparison.csv"
)

clean_compare_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)
display(clean_compare_df)

Saved: /content/drive/MyDrive/MLP/Project5/analysis_outputs/vit_baseline_vs_lazystrike_clean_comparison.csv


,model_name,fake_total,fake_TP_to_FN,fake_FN_to_TP,fake_TP_to_FN_ratio_percent,mean_prob_fake_drop_fake_images,interpretation
0,ViT-B/16,6000,102,19,1.700000,0.016788,Baseline
1,ViT-B/16 + LazyStrike-k1,6000,136,12,2.266667,0.019980,"Higher fake sensitivity, but larger JPEG-induc..."


In [9]:
# ============================================================
# Save LazyStrike fake TP->FN candidate cases
# ============================================================

lazy_cases = paired[paired["transition"] == "fake_TP_to_FN"].copy()
lazy_cases = lazy_cases.sort_values("prob_fake_drop", ascending=False)

save_path = os.path.join(
    analysis_dir,
    "vit_b16_lazystrike_k1_fake_TP_to_FN_cases.csv"
)

lazy_cases.to_csv(save_path, index=False, encoding="utf-8-sig")

print("Saved:", save_path)

display(
    lazy_cases[
        [
            "image_id_orig",
            "filename_orig",
            "filename_jpeg",
            "prob_fake_orig",
            "prob_fake_jpeg",
            "prob_fake_drop",
            "transition",
        ]
    ].head(10)
)

Saved: /content/drive/MyDrive/MLP/Project5/analysis_outputs/vit_b16_lazystrike_k1_fake_TP_to_FN_cases.csv


,image_id_orig,filename_orig,filename_jpeg,prob_fake_orig,prob_fake_jpeg,prob_fake_drop,transition
3020,3021,3021.jpg,3021.jpg,0.978206,0.016045,0.962161,fake_TP_to_FN
2340,2341,2341.png,2341.jpg,0.969941,0.021454,0.948487,fake_TP_to_FN
1101,1102,1102.png,1102.jpg,0.966297,0.036704,0.929592,fake_TP_to_FN
1468,1469,1469.png,1469.jpg,0.999902,0.072952,0.926950,fake_TP_to_FN
3634,3635,3635.jpg,3635.jpg,0.952281,0.033417,0.918864,fake_TP_to_FN
4160,4161,4161.jpg,4161.jpg,0.932304,0.013929,0.918375,fake_TP_to_FN
5493,5494,5494.jpg,5494.jpg,0.993287,0.076525,0.916762,fake_TP_to_FN
4246,4247,4247.jpg,4247.jpg,0.996105,0.113594,0.882511,fake_TP_to_FN
1033,1034,1034.png,1034.jpg,0.988631,0.114502,0.874129,fake_TP_to_FN
711,0712,0712.png,0712.jpg,0.998228,0.146076,0.852152,fake_TP_to_FN
